# Probabilidad y Estadística — Proyecto final
## Clasificador de máxima verosimilitud para MNIST

Carga y exploración del conjunto de datos MNIST de dígitos escritos a mano.

## Derivación analítica — Clasificador de máxima verosimilitud

---

### Notación

| Símbolo | Significado |
|---|---|
| $x \in \mathbb{R}^d$ | una imagen aplanada, $d = 28\times 28 = 784$ |
| $y \in \{0,\dots,9\}$ | etiqueta del dígito |
| $\mathcal{D} = \{(x^{(n)}, y^{(n)})\}_{n=1}^{N}$ | conjunto de entrenamiento, $N = 60\,000$ |
| $N_c$ | número de muestras de entrenamiento en la clase $c$ |

---

### 1. ¿Qué quiero construir?

Quiero una función que reciba una imagen y devuelva una clase:

$$f(x) = \hat{y}$$

Aquí $x$ es una imagen nueva, que el modelo no ha visto, y $\hat{y}$ es la clase predicha.
El sombrerito en $\hat{y}$ significa: *esta no necesariamente es la clase verdadera, es mi estimación.*

Como hay diez clases posibles, quiero escoger la más probable:

$$\hat{y} = \arg\max_{c \in \{0,\dots,9\}} P(y=c \mid x)$$

---

### 2. El teorema de Bayes

La cantidad $P(y=c \mid x)$ no se modela directamente. Usamos Bayes:

$$P(y=c \mid x) = \frac{P(y=c)\,p(x \mid y=c)}{p(x)}$$

Bayes cambia la pregunta: en vez de *«dada la imagen, ¿qué tan probable es la clase?»*,
pregunta *«si yo supiera que la clase es $c$, ¿qué tan compatible sería observar una imagen como $x$?»*.

Como $p(x)$ no depende de $c$:

$$\hat{y} = \arg\max_c P(y=c)\,p(x \mid y=c)$$

Aplicando logaritmo:

$$\hat{y} = \arg\max_c \bigl[\log P(y=c) + \log p(x \mid y=c)\bigr]$$

---

### 3. Suposición de modelado: distribución normal multivariada

Suponemos que, dentro de cada clase, las imágenes se distribuyen aproximadamente como una normal multivariada:

$$x \mid y=c \sim \mathcal{N}(\mu_c, \Sigma_c)$$

No decimos que $y$ sigue una normal. La clase $y$ es discreta ($y \in \{0,1,\dots,9\}$).
Lo que modelamos como normal es la imagen $x$ cuando ya sabemos la clase.

La densidad es:

$$p(x \mid y=c) = \frac{1}{(2\pi)^{d/2}\,|\Sigma_c|^{1/2}}
\exp\!\Bigl(-\tfrac12 (x-\mu_c)^{\mathsf{T}}
\Sigma_c^{-1}(x-\mu_c)\Bigr)$$

---

### 4. Estimación de máxima verosimilitud

Los parámetros $\mu_c$ y $\Sigma_c$ no se conocen; los estimamos desde los datos.

La media de la clase $c$ se estima promediando todas las imágenes que pertenecen a esa clase:

$$\hat{\mu}_c = \frac{1}{N_c} \sum_{i:\,y_i=c} x_i$$

La covarianza de la clase $c$ se estima mirando cómo se dispersan las imágenes alrededor de su media:

$$\hat{\Sigma}_c = \frac{1}{N_c} \sum_{i:\,y_i=c} (x_i - \hat{\mu}_c)(x_i - \hat{\mu}_c)^{\mathsf{T}}$$

La probabilidad previa (prior) se estima contando clases:

$$\hat{P}(y=c) = \frac{N_c}{N}$$

> **Regularización.** Como $d=784$ es grande, $\hat{\Sigma}_c$ puede estar mal condicionada
> o ser singular. Añadimos una perturbación diagonal: $\hat{\Sigma}_c^{\text{reg}} = \hat{\Sigma}_c + \varepsilon \mathbf{I}_d$, $\varepsilon = 10^{-4}$.

---

### 5. Regla de clasificación

Sustituyendo la densidad normal en la regla de decisión:

$$\hat{y} = \arg\max_c \bigl[\log \hat{P}(y=c) - \tfrac12 \log|\hat{\Sigma}_c| - \tfrac12 (x-\hat{\mu}_c)^{\mathsf{T}}\hat{\Sigma}_c^{-1}(x-\hat{\mu}_c)\bigr]$$

Cada término tiene interpretación:
- $\log \hat{P}(y=c)$: premia clases más frecuentes.
- $-\frac12 \log|\hat{\Sigma}_c|$: penaliza clases con mucha dispersión.
- $-\frac12 (x-\hat{\mu}_c)^{\mathsf{T}}\hat{\Sigma}_c^{-1}(x-\hat{\mu}_c)$: mide qué tan lejos está $x$ de la media de la clase (distancia de Mahalanobis).

---

### 6. Resumen de implementación

1. **Normalizar** intensidades de píxeles a $[0,1]$.
2. Para cada clase $c=0,\dots,9$:
   - $\hat{\mu}_c$ ← media muestral (MLE)
   - $\hat{\Sigma}_c$ ← covarianza muestral + $10^{-4}\mathbf{I}$ (MLE, regularizada)
   - Pre‑computar $\hat{\Sigma}_c^{-1}$ y $\log|\hat{\Sigma}_c|$
3. Para cada imagen de prueba $x$:
   - Calcular $M_c(x)$ para cada $c$
   - Predecir $\hat{y} = \arg\!\min_c \bigl[\tfrac12\log|\hat{\Sigma}_c| + \tfrac12 M_c(x) - \log N_c\bigr]$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='liac-arff')

X = X.astype(np.float32)
y = y.astype(np.int64)

print(f'Dimensiones de las imágenes: {X.shape}')
print(f'Dimensiones de las etiquetas: {y.shape}')
print(f'Rango de píxeles: [{X.min()}, {X.max()}]')
print(f'Etiquetas: {np.unique(y)}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Etiqueta: {y[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
unique, counts = np.unique(y, return_counts=True)
plt.bar(unique, counts)
plt.xlabel('Dígito')
plt.ylabel('Cantidad')
plt.title('Distribución de etiquetas')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=42, stratify=y)

print(f'Entrenamiento: {X_train.shape[0]} muestras')
print(f'Prueba:         {X_test.shape[0]} muestras')

In [ ]:
# --------------------------------------------------------------------
# 1. Normalizar los datos: cada imagen en R^784 con píxeles en [0,1]
# --------------------------------------------------------------------
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0

print(f'Rango píxeles entrenamiento: [{X_train_norm.min():.3f}, {X_train_norm.max():.3f}]')
print(f'Rango píxeles prueba:         [{X_test_norm.min():.3f},  {X_test_norm.max():.3f}]')

In [ ]:
# --------------------------------------------------------------------
# 2. Estimación de máxima verosimilitud de los parámetros
#    Ajustar una gaussiana multivariada condicional a cada dígito 0–9
#    p(x | y=c) = N(x ; μ_c , Σ_c)
# --------------------------------------------------------------------

classes    = np.unique(y_train)
n_classes  = len(classes)
n_features = X_train_norm.shape[1]          # 784

reg = 1e-4   # regularizar Σ para evitar singularidad

means   = np.zeros((n_classes, n_features))
covs    = np.zeros((n_classes, n_features, n_features))
priors  = np.zeros(n_classes)

for i, c in enumerate(classes):
    X_c        = X_train_norm[y_train == c]
    means[i]   = X_c.mean(axis=0)
    covs[i]    = np.cov(X_c, rowvar=False) + reg * np.eye(n_features)
    priors[i]  = len(X_c) / len(y_train)

print(f'Dimensión de medias:   {means.shape}')
print(f'Dimensión de covarianzas: {covs.shape}')
print(f'Priors:        {priors.round(4)}')
print(f'Suma de priors: {priors.sum():.4f}')

In [ ]:
# --------------------------------------------------------------------
# 3. Pre‑computar Σ_c^{-1} y log|Σ_c|, luego definir el predictor MAP
# --------------------------------------------------------------------

cov_invs      = np.zeros_like(covs)
log_det_covs  = np.zeros(n_classes)

for i in range(n_classes):
    cov_invs[i]                = np.linalg.inv(covs[i])
    _, log_det_covs[i]         = np.linalg.slogdet(covs[i])


def predict(X, means, cov_invs, log_det_covs, priors):
    """Clasificar cada muestra usando la regla MAP.
    Maximizar  log p(x | μ_c, Σ_c) + log P(c)  sobre las clases c.
    """
    n_samples    = X.shape[0]
    log_posterior = np.zeros((n_samples, n_classes))

    for i in range(n_classes):
        diff        = X - means[i]          # (n_samples, n_features)
        mahalanobis = np.sum(diff @ cov_invs[i] * diff, axis=1)

        log_posterior[:, i] = (
            -0.5 * (n_features * np.log(2 * np.pi) + log_det_covs[i] + mahalanobis)
            + np.log(priors[i])
        )

    return np.argmax(log_posterior, axis=1), log_posterior


y_pred, log_post = predict(X_test_norm, means, cov_invs, log_det_covs, priors)
print(f'Dimensiones de predicciones: {y_pred.shape}')

In [ ]:
# --------------------------------------------------------------------
# 4. Evaluación
# --------------------------------------------------------------------

accuracy = np.mean(y_pred == y_test)
print(f'Exactitud en prueba: {accuracy:.4f}  ({accuracy*100:.2f} %)')

print('\nExactitud por clase:')
for c in classes:
    mask = y_test == c
    if mask.sum() > 0:
        acc_c = np.mean(y_pred[mask] == c)
        print(f'  Dígito {c}: {acc_c:.4f}  ({mask.sum()} muestras)')

In [ ]:
# --------------------------------------------------------------------
# 5. Matrices de covarianza por clase
# --------------------------------------------------------------------

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, c in enumerate(classes):
    ax = axes.flat[i]
    # Mostrar la submatriz 28x28 de la covarianza (píxeles en la misma posición)
    cov_diag = np.diag(covs[i]).reshape(28, 28)
    im = ax.imshow(cov_diag, cmap='hot', interpolation='nearest')
    ax.set_title(f'Dígito {c}', fontsize=11)
    ax.axis('off')

fig.suptitle('Varianza por píxel de cada clase (diagonal de $\\hat{\\Sigma}_c$)', fontsize=13)
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Varianza')
plt.tight_layout()
plt.show()

In [ ]:
# --------------------------------------------------------------------
# 6. Animación — inferencias correctas (con media de clase y confianza)
# --------------------------------------------------------------------
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

correct_mask   = y_pred == y_test
correct_idx    = np.where(correct_mask)[0]

n_show = min(80, len(correct_idx))
show   = correct_idx[:n_show]

# Calcular confianzas (softmax sobre log_post)
from scipy.special import softmax
confidences_all = np.max(softmax(log_post, axis=1), axis=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                         gridspec_kw={'width_ratios': [1, 1, 1]})

ax_img  = axes[0]
ax_mean = axes[1]
ax_bar  = axes[2]

img_disp  = ax_img.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
mean_disp = ax_mean.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_img.set_title('Imagen de prueba')
ax_mean.set_title('Media de la clase predicha')
ax_img.axis('off')
ax_mean.axis('off')

# Barras de probabilidad posterior
bar_positions = np.arange(n_classes)
bars = ax_bar.bar(bar_positions, np.zeros(n_classes), color='steelblue')
ax_bar.set_xticks(bar_positions)
ax_bar.set_xlabel('Clase')
ax_bar.set_ylabel('P(y=c | x)')
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title('Probabilidad posterior')

conf_text = fig.suptitle('', fontsize=12, y=0.98)

def update(frame):
    i = show[frame]
    img_disp.set_data(X_test_norm[i].reshape(28, 28))
    pred_class = y_pred[i]
    mean_disp.set_data(means[pred_class].reshape(28, 28))
    probs = softmax(log_post[i:i+1], axis=1)[0]
    for b, p in zip(bars, probs):
        b.set_height(p)
        b.set_color('crimson' if b.get_x() == pred_class else 'steelblue')
    conf = confidences_all[i]
    conf_text.set_text(f'Verdadero: {y_test[i]}   |   Predicho: {pred_class}   |   Confianza: {conf:.2%}')
    return img_disp, mean_disp, conf_text

ani = FuncAnimation(fig, update, frames=n_show, interval=300, blit=False, repeat=True)
plt.tight_layout()
plt.close()
HTML(ani.to_jshtml())

In [ ]:
# --------------------------------------------------------------------
# 7. Animación — errores de predicción (con media de clase y confianza)
# --------------------------------------------------------------------

error_mask = y_pred != y_test
error_idx  = np.where(error_mask)[0]

n_show = min(80, len(error_idx))
show   = error_idx[:n_show]

fig, axes = plt.subplots(1, 3, figsize=(12, 4),
                         gridspec_kw={'width_ratios': [1, 1, 1]})

ax_img  = axes[0]
ax_mean = axes[1]
ax_bar  = axes[2]

img_disp  = ax_img.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
mean_disp = ax_mean.imshow(np.zeros((28, 28)), cmap='gray', vmin=0, vmax=1)
ax_img.set_title('Imagen de prueba')
ax_mean.set_title('Media de la clase predicha')
ax_img.axis('off')
ax_mean.axis('off')

bar_positions = np.arange(n_classes)
bars = ax_bar.bar(bar_positions, np.zeros(n_classes), color='steelblue')
ax_bar.set_xticks(bar_positions)
ax_bar.set_xlabel('Clase')
ax_bar.set_ylabel('P(y=c | x)')
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title('Probabilidad posterior')

conf_text = fig.suptitle('', fontsize=12, color='red', fontweight='bold', y=0.98)

def update(frame):
    i = show[frame]
    img_disp.set_data(X_test_norm[i].reshape(28, 28))
    pred_class = y_pred[i]
    mean_disp.set_data(means[pred_class].reshape(28, 28))
    probs = softmax(log_post[i:i+1], axis=1)[0]
    for b, p in zip(bars, probs):
        b.set_height(p)
        b.set_color('crimson' if b.get_x() == pred_class else 'steelblue')
    conf = confidences_all[i]
    conf_text.set_text(f'Verdadero: {y_test[i]}   |   Predicho (ERROR): {pred_class}   |   Confianza: {conf:.2%}')
    return img_disp, mean_disp, conf_text

ani = FuncAnimation(fig, update, frames=n_show, interval=400, blit=False, repeat=True)
plt.tight_layout()
plt.close()
HTML(ani.to_jshtml())